# Converting schema to clean human-readable version

## Read the schema (of unmapped entity)

In [1]:
import yaml
from typing import Any, Dict

In [2]:
import yaml

def determine_type_prefix(field_info):
    """Returns the bracketed type prefix string using Pythonic naming conventions, safely handling union type lists."""
    field_type = field_info.get("type", "any")
    
    # Handle multi-type lists (e.g., [number, string, boolean])
    if isinstance(field_type, list):
        type_strings = []
        for t in field_type:
            if t == "object":
                type_strings.append("Dictionary")
            elif t == "array":
                type_strings.append("List")
            else:
                type_strings.append(str(t).capitalize())
        return "/".join(type_strings)

    # Handle standard single string types
    if field_type == "object":
        return "Dictionary"
    elif field_type == "array":
        item_type = field_info.get("items", {}).get("type", "any")
        
        # Handle multi-type lists inside an array item configuration
        if isinstance(item_type, list):
            item_str = "/".join(str(t).capitalize() for t in item_type)
            return f"List of {item_str}s"
        elif item_type == "object":
            return "List of Dictionaries"
        return f"List of {item_type.capitalize()}s"
        
    return f"{field_type.capitalize()}"


def parse_properties(schema, required_fields, indent_level=4):
    """Recursively processes properties into a flat blueprint, inserting requirements into the description text."""
    lines = []
    space = " " * indent_level
    properties = schema.get("properties", {})

    for field_name, field_info in properties.items():
        field_type = field_info.get("type", "any")
        description = field_info.get("description", "No description provided.")
        
        # 1. Determine requirement status prefix
        req_prefix = "Required; " if field_name in required_fields else ""
        
        # 2. Determine pythonic data type prefix
        type_prefix = determine_type_prefix(field_info)

        # 3. Assemble the full text string
        desc_text = f"[{req_prefix}{type_prefix}] {description}"
        if "examples" in field_info:
            desc_text += f" Suggested: {field_info['examples']}"

        # Handle Nested Dictionaries
        if field_type == "object" and "properties" in field_info:
            inner_required = field_info.get("required", [])
            lines.append(f"{space}{field_name}: # {desc_text}")
            lines.extend(
                parse_properties(field_info, inner_required, indent_level + 4)
            )

        # Handle Lists of Dictionaries
        elif field_type == "array" and "properties" in field_info.get("items", {}):
            inner_required = field_info["items"].get("required", [])
            lines.append(f"{space}{field_name}: # {desc_text}")
            lines.extend(
                parse_properties(field_info["items"], inner_required, indent_level + 4)
            )

        # Handle primitive or mixed data types (Strings, Numbers, multi-type arrays)
        else:
            lines.append(f'{space}{field_name}: "{desc_text}"')

    return lines


def convert_schema_to_clean_blueprint(input_yaml_path, output_yaml_path):
    """Reads a JSON Schema and outputs a clean, pythonic comment-style documentation blueprint."""
    with open(input_yaml_path, "r", encoding="utf-8") as f:
        schema_data = yaml.safe_load(f)

    title = schema_data.get("title", "SchemaDocumentation")
    global_desc = schema_data.get("description", "No global description provided.")
    top_required = schema_data.get("required", [])

    output_lines = [
        f"{title}: # {global_desc}",
    ]

    body_lines = parse_properties(schema_data, top_required, indent_level=4)
    output_lines.extend(body_lines)

    with open(output_yaml_path, "w", encoding="utf-8") as f:
        f.write("\n".join(output_lines) + "\n")

    print(f"Successfully generated clean human blueprint at: {output_yaml_path}")

In [3]:
## read all the yaml files in the folder above one level

import os
import yaml
from pathlib import Path

# Get the parent directory
parent_dir = Path.cwd().parent

# Find all YAML files
yaml_files = list(parent_dir.glob('*.yaml')) + list(parent_dir.glob('*.yml'))

for input_path in yaml_files:
    # Use the same filename for the output (overwrites original)
    output_path = input_path.name
    
    try:
        # Call your conversion function
        convert_schema_to_clean_blueprint(str(input_path), str(output_path))
        print(f"Successfully processed: {input_path.name}")
    except Exception as e:
        print(f"Error processing {input_path.name}: {e}")

In [4]:
## convert all the yaml files in the 'path_schema' folder and their subfolders into a clean human blueprint in the 'path_schema_human' folder

path_schema = r'../schema'
path_schema_simple = r'../schema_human'

src_root = Path(path_schema)
dst_root = Path(path_schema_simple)

# Ensure destination root exists
dst_root.mkdir(parents=True, exist_ok=True)

# Find all YAML files recursively
yaml_files = list(src_root.rglob("*.yaml")) + list(src_root.rglob("*.yml"))

processed = 0
failed = 0

for input_path in yaml_files:
    # Preserve subfolder structure in destination
    relative_path = input_path.relative_to(src_root)
    output_path = dst_root / relative_path
    output_path.parent.mkdir(parents=True, exist_ok=True)

    try:
        convert_schema_to_clean_blueprint(str(input_path), str(output_path))
        print(f"Successfully processed: {input_path} -> {output_path}")
        processed += 1
    except Exception as e:
        print(f"Error processing {input_path}: {e}")
        failed += 1

print(f"\nDone. Processed: {processed}, Failed: {failed}")


Successfully generated clean human blueprint at: ..\schema_human\linked_entity.yaml
Successfully processed: ..\schema\linked_entity.yaml -> ..\schema_human\linked_entity.yaml
Successfully generated clean human blueprint at: ..\schema_human\unmapped_entity.yaml
Successfully processed: ..\schema\unmapped_entity.yaml -> ..\schema_human\unmapped_entity.yaml
Successfully generated clean human blueprint at: ..\schema_human\controlled_vocabulary\attribute.yaml
Successfully processed: ..\schema\controlled_vocabulary\attribute.yaml -> ..\schema_human\controlled_vocabulary\attribute.yaml
Successfully generated clean human blueprint at: ..\schema_human\controlled_vocabulary\capacity_scope.yaml
Successfully processed: ..\schema\controlled_vocabulary\capacity_scope.yaml -> ..\schema_human\controlled_vocabulary\capacity_scope.yaml
Successfully generated clean human blueprint at: ..\schema_human\controlled_vocabulary\carrier.yaml
Successfully processed: ..\schema\controlled_vocabulary\carrier.yaml ->